# Fatigue Example 5 — Multiaxial Fatigue and Stress Concentration (Computational Workflow by M.A.Z. Chowhdhury)

**Shaft with a transverse hole**, subjected to cyclic **released** bending and torsion.

**Material:** 4320 steel — $S_u = 150\ \mathrm{ksi}$, $S_{yp} = 120\ \mathrm{ksi}$, $e = 15\%$ (from the given $S$–$N$ curve)

**Goal:** determine the number of cycles to failure (infinite-life check) and the governing safety factors.

This notebook implements the equations the worked solution and evaluates it numerically with Python, so you can sanity-check the arithmetic and quickly re-run the problem with different geometry or loads.


In [16]:
from IPython.display import Image, display
import base64

import numpy as np

# ---------------------------------------------------------------
# Given data
# ---------------------------------------------------------------
D = 1.25          # in, shaft (gross) diameter
d = 0.25          # in, transverse hole diameter

T_max = 8300.0     # lb-in, applied torque (max)
T_min = 0.0        # lb-in, applied torque (min)  -> released cycle
M_max = 3700.0      # lb-in, applied bending moment (max)
M_min = 0.0         # lb-in, applied bending moment (min) -> released cycle

Su  = 150e3        # psi, ultimate tensile strength
Syp = 120e3         # psi, yield strength
e   = 0.15          # elongation (informational, not used numerically below)

print(f"D     = {D} in")
print(f"d     = {d} in")
print(f"T_max = {T_max} lb-in,  T_min = {T_min} lb-in")
print(f"M_max = {M_max} lb-in,  M_min = {M_min} lb-in")
print(f"Su    = {Su/1e3:.0f} ksi")
print(f"Syp   = {Syp/1e3:.0f} ksi")


shaft_url = "https://raw.githubusercontent.com/arshadzahangirchowdhury/Images/main/Fatigue_ex6_Shaft.jpg"
sn_url = "https://raw.githubusercontent.com/arshadzahangirchowdhury/Images/main/Fatigue_ex6_SN.jpg"
stress_conc_url = "https://raw.githubusercontent.com/arshadzahangirchowdhury/Images/main/Collins_Fig5_6_a.png"
notch_sensitivity_url = "https://raw.githubusercontent.com/arshadzahangirchowdhury/Images/main/Collins_Fig_5_46.png"

display(Image(url=shaft_url, width=500))
display(Image(url=sn_url, width=500))
display(Image(url=stress_conc_url, width=1000))
display(Image(url=notch_sensitivity_url, width=1000))



D     = 1.25 in
d     = 0.25 in
T_max = 8300.0 lb-in,  T_min = 0.0 lb-in
M_max = 3700.0 lb-in,  M_min = 0.0 lb-in
Su    = 150 ksi
Syp   = 120 ksi


## Step 1 — Nominal maximum and minimum stresses

Bending:
$$\sigma_{xx-nom}^{\max} = \frac{M_{\max}}{\dfrac{\pi D^3}{32} - \dfrac{dD^2}{6}} \qquad \sigma_{xx-nom}^{\min}=0$$

Torsion (shear):
$$\sigma_{xy-nom}^{\max}=0 \qquad \sigma_{xy-nom}^{\min} = \frac{-T_{\max}}{\dfrac{\pi D^3}{16} - \dfrac{dD^2}{6}}$$

*(Sign convention on $T_{\max}$ depends on the choice of axes — here it's negative.)*


In [17]:


sigma_xx_nom_max = M_max / ((np.pi * D**3) / 32 - (d * D**2) / 6 )
sigma_xx_nom_min = 0.0

sigma_xy_nom_max = 0.0
sigma_xy_nom_min = -T_max / ((np.pi * D**3) / 16 - (d * D**2) / 6)

print(f"sigma_xx_nom_max = {sigma_xx_nom_max:,.0f} psi   (ref: 29,216 psi)")
print(f"sigma_xx_nom_min = {sigma_xx_nom_min:,.0f} psi")
print(f"sigma_xy_nom_max = {sigma_xy_nom_max:,.0f} psi")
print(f"sigma_xy_nom_min = {sigma_xy_nom_min:,.0f} psi   (ref: -26,069 psi)")


sigma_xx_nom_max = 29,216 psi   (ref: 29,216 psi)
sigma_xx_nom_min = 0 psi
sigma_xy_nom_max = 0 psi
sigma_xy_nom_min = -26,069 psi   (ref: -26,069 psi)


## Step 2 — Nominal alternating and mean stress components

$$\sigma_{a} = \frac{\sigma^{\max}-\sigma^{\min}}{2} \qquad \sigma_{m} = \frac{\sigma^{\max}+\sigma^{\min}}{2}$$

Applied to both bending ($xx$) and torsion ($xy$).


In [18]:
sigma_xx_nom_a = (sigma_xx_nom_max - sigma_xx_nom_min) / 2
sigma_xx_nom_m = (sigma_xx_nom_max + sigma_xx_nom_min) / 2

sigma_xy_nom_a = (sigma_xy_nom_max - sigma_xy_nom_min) / 2
sigma_xy_nom_m = (sigma_xy_nom_max + sigma_xy_nom_min) / 2

print(f"sigma_xx_nom_a = {sigma_xx_nom_a:,.0f} psi   (ref: 14,608 psi)")
print(f"sigma_xx_nom_m = {sigma_xx_nom_m:,.0f} psi   (ref: 14,608 psi)")
print(f"sigma_xy_nom_a = {sigma_xy_nom_a:,.1f} psi  (ref: -13,034.5 psi)")
print(f"sigma_xy_nom_m = {sigma_xy_nom_m:,.1f} psi  (ref: -13,034.5 psi)")


sigma_xx_nom_a = 14,608 psi   (ref: 14,608 psi)
sigma_xx_nom_m = 14,608 psi   (ref: 14,608 psi)
sigma_xy_nom_a = 13,034.3 psi  (ref: -13,034.5 psi)
sigma_xy_nom_m = -13,034.3 psi  (ref: -13,034.5 psi)


## Step 3 — Fatigue stress concentration factors $k_f$

From the stress concentration chart (Fig. 5.6a-type Peterson chart) with $r = d/2 = 0.125$ in and $d/D = 0.25/1.25 = 0.2$:

$$k_{tb} = 2.03 \quad (\text{bending}), \qquad k_{tt} = 1.5 \quad (\text{torsion})$$

Notch sensitivity (read off notch sensitivity $q$-chart for 4320 steel, $S_u = 150$ ksi):

$$q_{bending} = 0.91, \qquad q_{torsion} = 0.93$$

Fatigue stress concentration factor:
$$k_f = q\,(k_t - 1) + 1$$

$k_f$ is applied **only to the alternating stress component**:
$$\sigma_{xx-a} = k_{fb}\,\sigma_{xx-nom-a}, \qquad \sigma_{xy-a} = k_{ft}\,\sigma_{xy-nom-a}$$


In [19]:
# Chart-read values
ktb, ktt = 2.03, 1.5
q_bend, q_tors = 0.91, 0.93

kfb = q_bend * (ktb - 1) + 1
kft = q_tors * (ktt - 1) + 1

sigma_xx_a = kfb * sigma_xx_nom_a
sigma_xy_a = kft * sigma_xy_nom_a   # sign carried through from sigma_xy_nom_a

print(f"k_fb = {kfb:.3f}   (ref: 1.94)")
print(f"k_ft = {kft:.3f}   (ref: 1.47)")
print()
print(f"sigma_xx_a = {sigma_xx_a:,.0f} psi   (ref: 28,340 psi)")
print(f"sigma_xy_a = {sigma_xy_a:,.0f} psi   (ref: -19,161 psi)")

# Mean stresses are carried through WITHOUT the Kf (per this solution's convention)
sigma_xx_m = sigma_xx_nom_m
sigma_xy_m = sigma_xy_nom_m
print()
print(f"sigma_xx_m = {sigma_xx_m:,.1f} psi")
print(f"sigma_xy_m = {sigma_xy_m:,.1f} psi")


k_fb = 1.937   (ref: 1.94)
k_ft = 1.465   (ref: 1.47)

sigma_xx_a = 28,300 psi   (ref: 28,340 psi)
sigma_xy_a = 19,095 psi   (ref: -19,161 psi)

sigma_xx_m = 14,607.9 psi
sigma_xy_m = -13,034.3 psi


## Step 4 — Combine components with the Distortion Energy Theory (DET / von Mises)

For a shaft surface element ($\sigma_{yy}=\sigma_{zz}=\sigma_{xz}=\sigma_{yz}=0$), the general DET equivalent-stress expression

$$\sigma_{eq}=\frac{1}{\sqrt2}\Big[(\sigma_{xx}-\sigma_{yy})^2+(\sigma_{yy}-\sigma_{zz})^2+(\sigma_{zz}-\sigma_{xx})^2+6\big(\sigma_{xy}^2+\sigma_{xz}^2+\sigma_{yz}^2\big)\Big]^{1/2}$$

reduces to

$$\sigma_{eq}=\frac{1}{\sqrt2}\Big[2\sigma_{xx}^2+6\sigma_{xy}^2\Big]^{1/2}$$

applied separately to the **alternating** and **mean** stress states.




In [20]:
def sigma_eq_DET(sxx, sxy):
    return (1/np.sqrt(2)) * np.sqrt(2*sxx**2 + 6*sxy**2)

# --- Alternating equivalent stress ---
sigma_eq_a = sigma_eq_DET(sigma_xx_a, sigma_xy_a)
print(f"sigma_eq_a = {sigma_eq_a:,.0f} psi   (ref: 43,642 psi)")

# --- Mean equivalent stress, using the correct sigma_xy_m = -13,034.5 psi ---
sigma_eq_m = sigma_eq_DET(sigma_xx_m, sigma_xy_m)
print(f"sigma_eq_m (sigma_xy_m = {sigma_xy_m:,.1f} psi) = {sigma_eq_m:,.0f} psi   (ref: 26,890 psi)")


sigma_eq_a = 43,529 psi   (ref: 43,642 psi)
sigma_eq_m (sigma_xy_m = -13,034.3 psi) = 26,890 psi   (ref: 26,890 psi)


## Step 5 — Modified Goodman Theory (MGT) and infinite-life check

Convert the alternating/mean pair into a single fully-reversed "completely-reversed" equivalent stress:

$$\sigma_{eq-CR} = \frac{\sigma_{eq-a}}{1-\dfrac{\sigma_{eq-m}}{S_{ut}}}$$

Compare $\sigma_{eq-CR}/S_{ut}$ against the fatigue-strength fraction $S_f/S_{ut}$ read off the $S$–$N$ curve (given as $0.72$ here) to decide infinite vs. finite life, and compute the safety factor guarding against fatigue failure:

$$n_f = \frac{S_f}{\sigma_{eq-CR}}$$


In [21]:
sigma_eq_CR = sigma_eq_a / (1 - sigma_eq_m / Su)
print(f"sigma_eq_CR = {sigma_eq_CR:,.0f} psi   (ref: 53,174 psi)")

ratio = sigma_eq_CR / Su
print(f"sigma_eq_CR / Su = {ratio:.3f}  ({ratio*100:.1f} %)")

# From the given S-N curve, at "infinite life" (~1e6-1e8 cycles):
Sf_over_Sut = 0.72
Sf = Sf_over_Sut * Su
print(f"Sf/Su = {Sf_over_Sut}  ->  Sf = {Sf/1e3:.0f} ksi")

if sigma_eq_CR < Sf:
    life_verdict = "INFINITE LIFE (sigma_eq_CR < Sf)"
else:
    life_verdict = "FINITE LIFE (sigma_eq_CR >= Sf) -- would need to read N off the S-N curve"
print(f"\nVerdict: {life_verdict}")

nf = Sf / sigma_eq_CR
print(f"\nFatigue safety factor  n_f = Sf / sigma_eq_CR = {nf:.2f}   (ref: ~2.03)")


sigma_eq_CR = 53,037 psi   (ref: 53,174 psi)
sigma_eq_CR / Su = 0.354  (35.4 %)
Sf/Su = 0.72  ->  Sf = 108 ksi

Verdict: INFINITE LIFE (sigma_eq_CR < Sf)

Fatigue safety factor  n_f = Sf / sigma_eq_CR = 2.04   (ref: ~2.03)


## Step 6 — Static yield check (DET) at the maximum load

Using the **maximum nominal** stresses:

$$\sigma_{eq}^{\max}=\frac{1}{\sqrt2}\Big[2(\sigma_{xx-nom}^{\max})^2+6(\sigma_{xy-nom}^{\max\ \text{or}\ \min})^2\Big]^{1/2}$$

Compare against $S_{yp}$ and compute the yield safety factor:
$$n_y = \frac{S_{yp}}{\sigma_{eq}^{\max}}$$


In [14]:
sigma_eq_max = sigma_eq_DET(sigma_xx_nom_max, sigma_xy_nom_min)
print(f"sigma_eq_max = {sigma_eq_max:,.0f} psi   (ref: 53,781 psi)")

no_yield = sigma_eq_max < Syp
print(f"sigma_eq_max < Syp ({Syp/1e3:.0f} ksi)?  ->  {no_yield}  (No yield expected)" )

ny = Syp / sigma_eq_max
print(f"\nYield safety factor  n_y = Syp / sigma_eq_max = {ny:.2f}   (ref: ~2.23)")

local_yield_note = "n_y > max(k_tb, k_tt) -> local (notch-tip) yielding is NOT expected either."
print(local_yield_note)


sigma_eq_max = 53,780 psi   (ref: 53,781 psi)
sigma_eq_max < Syp (120 ksi)?  ->  True  (No yield expected)

Yield safety factor  n_y = Syp / sigma_eq_max = 2.23   (ref: ~2.23)
n_y > max(k_tb, k_tt) -> local (notch-tip) yielding is NOT expected either.


## Summary


In [15]:
print("="*60)
print("SUMMARY OF RESULTS")
print("="*60)
print(f"{'Nominal bending stress, max':38s} = {sigma_xx_nom_max:10,.0f} psi")
print(f"{'Nominal torsional stress, min':38s} = {sigma_xy_nom_min:10,.0f} psi")
print(f"{'k_fb (bending)':38s} = {kfb:10.3f}")
print(f"{'k_ft (torsion)':38s} = {kft:10.3f}")
print(f"{'Equivalent alternating stress':38s} = {sigma_eq_a:10,.0f} psi")
print(f"{'Equivalent mean stress':38s} = {sigma_eq_m:10,.0f} psi")
print(f"{'Goodman-corrected stress':38s} = {sigma_eq_CR:10,.0f} psi")
print(f"{'Fatigue safety factor n_f':38s} = {nf:10.2f}")
print(f"{'Static equivalent stress (max load)':38s} = {sigma_eq_max:10,.0f} psi")
print(f"{'Yield safety factor n_y':38s} = {ny:10.2f}")
print("="*60)
print(f"Life verdict: {life_verdict}")


SUMMARY OF RESULTS
Nominal bending stress, max            =     29,216 psi
Nominal torsional stress, min          =    -26,069 psi
k_fb (bending)                         =      1.937
k_ft (torsion)                         =      1.465
Equivalent alternating stress          =     43,529 psi
Equivalent mean stress                 =     26,890 psi
Goodman-corrected stress               =     53,037 psi
Fatigue safety factor n_f              =       2.04
Static equivalent stress (max load)    =     53,780 psi
Yield safety factor n_y                =       2.23
Life verdict: INFINITE LIFE (sigma_eq_CR < Sf)
